In [1]:

import cv2

# 1. 打开电脑默认摄像头
cap = cv2.VideoCapture(0)

# 2. 检查摄像头是否成功打开
if not cap.isOpened():
    print("摄像头打开失败！")
    exit()

print("摄像头启动成功！")
print("按 Q 键退出程序")

# 3. 循环读取摄像头画面
while True:

    ret, frame = cap.read()

    # 检查是否成功读取图像
    if not ret:
        print("读取摄像头画面失败！")
        break

    # 4. 显示摄像头画面
    cv2.imshow("Robot Vision Demo", frame)

    # 5. 按 Q 键退出
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 6. 释放摄像头资源
cap.release()

# 7. 关闭所有窗口
cv2.destroyAllWindows()


摄像头启动成功！
按 Q 键退出程序


In [1]:

import cv2
import numpy as np

# 1. 打开摄像头
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("摄像头打开失败！")
    exit()

print("摄像头启动成功！按 Q 退出")

# 2. 定义颜色范围（OpenCV 的 HSV）
color_ranges = {
    "GREEN": ([35, 60, 50], [85, 255, 255]),
    "BLUE":  ([90, 60, 50], [130, 255, 255]),
    "RED1":  ([0, 80, 50], [10, 255, 255]),
    "RED2":  ([170, 80, 50], [180, 255, 255]),
    "GRAY":  ([0, 0, 80], [180, 60, 220])
}

# 创建 5×5 的卷积核
kernel = np.ones((5,5), np.uint8)


while True:

    # 3. 读取摄像头画面
    ret, frame = cap.read()

    if not ret:
        print("读取画面失败！")
        break

    # 4. 转换为 HSV
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # 5. 分别识别颜色
    for color_name, (lower, upper) in color_ranges.items():

        lower = np.array(lower)
        upper = np.array(upper)

        mask = cv2.inRange(hsv, lower, upper)

        # 红色有两个范围，需要合并
        if color_name == "RED1":
            red_mask = mask
            continue

        if color_name == "RED2":
            red_mask = cv2.bitwise_or(red_mask, mask)
            mask = red_mask
            color_name = "RED"

        # 开运算：去除小白点
        mask = cv2.morphologyEx(
            mask,
            cv2.MORPH_OPEN,
            kernel
        )

        # 闭运算：填补小黑洞
        mask = cv2.morphologyEx(
            mask,
            cv2.MORPH_CLOSE,
            kernel
        )
        # 6. 查找符合颜色的轮廓
        contours, hierarchy = cv2.findContours(
            mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        for contour in contours:

            area = cv2.contourArea(contour)

            # 忽略太小的区域
            if area < 500:
                continue

            x, y, w, h = cv2.boundingRect(contour)

            # 计算目标中心点
            center_x = x + w // 2
            center_y = y + h // 2

            # 7. 绘制识别框
            cv2.rectangle(
                frame,
                (x, y),
                (x + w, y + h),
                (0, 255, 0),
                2
            )

            # 绘制目标中心点
            cv2.circle(
                frame,
                (center_x, center_y),
                5,
                (0, 0, 255),
                -1
            )

            # 显示目标信息
            info = f"X:{center_x} Y:{center_y} W:{w} H:{h}"

            cv2.putText(
                frame,
                info,
                (x, y + h + 25),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (255, 255, 255),
                2
            )
            cv2.putText(
                frame,
                color_name,
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

    # 8. 显示结果
    cv2.imshow("Robot Vision Demo", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


摄像头启动成功！按 Q 退出


In [5]:

import cv2

# 1. 打开摄像头
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("摄像头打开失败！")
    exit()

print("矩形检测启动成功，按 Q 退出")

while True:

    # 2. 读取摄像头画面
    ret, frame = cap.read()

    if not ret:
        print("读取画面失败！")
        break

    # 3. 灰度化
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # 4. CLAHE 局部对比度增强
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )
    enhanced = clahe.apply(gray)

    # 5. 高斯模糊，减少噪点
    blurred = cv2.GaussianBlur(
        enhanced,
        (5, 5),
        0
    )

    # 6. Canny 边缘检测
    edges = cv2.Canny(
        blurred,
        30,
        90
    )

    # 7. 创建 3×3 矩形结构元素
    kernel = cv2.getStructuringElement(
        cv2.MORPH_RECT,
        (3, 3)
    )

    # 8. 闭运算，连接较小的边缘断裂
    edges_closed = cv2.morphologyEx(
        edges,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=2
    )

    # 9. 寻找轮廓
    contours, hierarchy = cv2.findContours(
        edges_closed,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # 10. 初始化矩形数量
    rectangle_count = 0

    # 11. 遍历每一个轮廓
    for contour in contours:

        # 计算轮廓面积
        area = cv2.contourArea(contour)

        # 过滤太小的轮廓
        if area < 1000:
            continue

        # 计算轮廓周长
        perimeter = cv2.arcLength(contour, True)

        # 轮廓近似
        approx = cv2.approxPolyDP(
            contour,
            0.04 * perimeter,
            True
        )

        # 判断是否为凸多边形
        is_convex = cv2.isContourConvex(approx)

        # 获取外接矩形
        box_x, box_y, box_w, box_h = cv2.boundingRect(approx)

        # 绘制近似轮廓
        cv2.drawContours(
            frame,
            [approx],
            -1,
            (255, 0, 0),
            2
        )

        # 显示顶点数量和凸性
        info = f"Points:{len(approx)} Convex:{is_convex}"

        cv2.putText(
            frame,
            info,
            (box_x, max(box_y - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 255),
            2
        )

        # 四个顶点且为凸多边形：作为四边形候选
        if len(approx) == 4 and is_convex:

            # 数量加一
            rectangle_count += 1

            # 计算中心点坐标
            center_x = box_x + box_w // 2
            center_y = box_y + box_h // 2

            # 绘制外接矩形
            cv2.rectangle(
                frame,
                (box_x, box_y),
                (box_x + box_w, box_y + box_h),
                (0, 255, 0),
                2
            )

            # 绘制中心点
            cv2.circle(
                frame,
                (center_x, center_y),
                5,
                (0, 0, 255),
                -1
            )

            # 显示中心点坐标
            cv2.putText(
                frame,
                f"X:{center_x} Y:{center_y}",
                (box_x, max(box_y - 30, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 255, 0),
                2
            )

    # 12. 显示矩形数量
    cv2.putText(
        frame,
        f"Objects: {rectangle_count}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 255),
        2
    )

    # 13. 显示结果
    cv2.imshow("Rectangle Detection", frame)
    cv2.imshow("Edges", edges)
    # cv2.imshow("Closed Edges", edges_closed)
    # cv2.imshow("gray",gray)
    # 14. 按 Q 退出
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 15. 释放摄像头和窗口
cap.release()
cv2.destroyAllWindows()


矩形检测启动成功，按 Q 退出
